In [0]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StructType, IntegerType, StringType, StructField

@dlt.table(
    name = 'bronze_orders'
)
def bronze_orders():
    return spark.readStream.format('cloudFiles') \
        .option('cloudFiles.format', 'json') \
        .option('cloudFiles.inferSchema', 'true') \
        .option('cloudFiles.schemaHints', 'customer_id int, order_id int, order_timestamp timestamp') \
        .load('/Volumes/circuit_box/landing/circuit_volume/orders/') \
        .select('*', F.current_timestamp().alias('ingest_ts'), F.col('_metadata.file_path').alias('file_path'))


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import explode, from_json
import dlt

items_Schema =ArrayType(StructType([
        StructField("item_id", IntegerType()),
        StructField("name", StringType()),
        StructField("category", StringType()),
        StructField("price", IntegerType()),
        StructField("quantity", IntegerType()),
    ]
))

@dlt.table(
    name = 'silver_orders_cleanup'
)
@dlt.expect_all_or_fail( {'valid_customer_id': 'customer_id is not null', 'valid_order_id': 'order_id is not null'})
@dlt.expect_all( {'valid_payment_method'  : 'payment_method in ("Credit Card", "Bank Transfer", "PayPal")', 'valid_order_status' : 'order_status in ("Pending","Completed","Shipped","Cancelled")'})
def silver_clean_orders():
    df =  spark.readStream.table('Live.bronze_orders').withColumn('items', from_json(F.col('items'), items_Schema))
    df = df.select('customer_id', 'order_id', 'order_status', 'order_timestamp', explode('items').alias('item'), 'payment_method')
    df = df.select('customer_id', 'order_id', 'order_status', 'order_timestamp', 'item.*' ,'payment_method')
    return df



In [0]:
import dlt

@dlt.table(
    name = 'silver_orders',
    comment = 'This is silver order table',
    table_properties = {'quality' : 'silver'}
)
def silver_table():
    return spark.readStream.table('Live.silver_orders_cleanup').select("*")